# Particle Scanning + 3D CNN Refinement Pipeline
This notebook runs a two-stage particle workflow: DETR-style candidate generation followed by 3D CNN classification/refinement and optional adjusted-AUPR evaluation.

## Release Notes
- Choose exactly one target class configuration (ribosome or hsp60) in each section.
- Input tomograms are `.mrc`; labels are loaded from class-specific text files.
- Temporary stage-1 predictions are written to `./temp/*.csv`.
- Run cells in order from top to bottom.

In [ ]:
import os
import sys

import numpy as np
import pandas as pd
import torch

sys.path.append("../src")

import data
import modules
import postprocess
import utils

os.makedirs("./temp", exist_ok=True)

## 1) Configure Input Tomograms
Choose the tomograms to process. Keep only one class setup active at a time (ribosome or hsp60) so model paths, score columns, and labels stay consistent.

In [56]:
# tomograms for ribosome
tomo_paths = {
    "test1": "<pathto>/release_model/data_example/ribosome.mrc"
}
# tomograms for hsp60
# tomo_paths = {
#     "test2": "<pathto>/release_model/data_example/hsp60.mrc"
# }

## 2) Stage-1 Detection and Candidate Extraction
Run the detector on each tomogram and save raw slice-wise predictions to `./temp/*.csv`.
This stage favors recall; downstream sweep filtering and 3D CNN classification reduce false positives.

In [57]:
# Stage 1 detector (choose one model block).
# ribosome
model = utils.loadModel("<pathto>/release_model/ribosome", "last.ckpt")
model = model.eval()
# hsp60
# model = utils.loadModel("<pathto>/release_model/hsp60", "last.ckpt")
# model = model.eval()

# Run detector over tomograms and store raw candidate boxes/scores.
if torch.cuda.is_available():
    model = model.cuda(0)

for i in tomo_paths:
    dataset = data.TestDatasetMrc(
        tomo_paths[i],
        norm="hist",
        reshape=800,
        length_for_average=3,
        gap=1,
    )
    df = postprocess.generatedfBySlice(model, dataset, gap=1, columns=["ribosome", "None"] )
    df.to_csv(f"./temp/{i}.csv", index=False)

model at stage  stage 1
model with output classes 2
model receiving class weights tensor([1.0000, 0.6000])
using consistency regularization coef 0.5


/data/biosoftware/miniconda3/miniconda3/envs/tomognn/lib/python3.11/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")
/home/feity/cryoem/notebooks/../src/utils.py:1397: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will

incompatible parameters []
finish loading parameters
loading test dataset


100%|██████████| 500/500 [01:28<00:00,  5.62it/s]


finish stage1


Stage 2 slices: 100%|██████████| 500/500 [00:25<00:00, 19.61it/s]


## 3) Sweep Filtering to Proposal Centers
Convert dense detector outputs into sparse center proposals using probability and spatial sweep thresholds.
Tune `PROB_THRES` and `SWEEP_THRES` per class and dataset quality.

In [ ]:
# Stage 1 post-processing thresholds for center proposals.
PROB_THRES = 0.20
TOMOGRAM_SIZE_X = 1024
TOMOGRAM_SIZE_Y = 1024
TOMOGRAM_SIZE_Z = 500
NEGATIVE_DIS = 35.0
# ribosome
SWEEP_THRES = 15
# hsp60
# SWEEP_THRES = 10

# Set this to the detector class column you are using (e.g., "ribosome" or "hsp60").
SCORE_COLUMN = "ribosome"

df_predicts = []
for i in tomo_paths:
    print(f"Processing tomogram: {i}")
    subdf_path = f"./temp/{i}.csv"
    df_predicts.append(
        utils.sweep_to_find_prediction_centers(
            tomogram_name=i,
            subdf_csv_path=subdf_path,
            prob_threshold=PROB_THRES,
            tomogram_size_x=TOMOGRAM_SIZE_X,
            tomogram_size_y=TOMOGRAM_SIZE_Y,
            tomogram_size_z=TOMOGRAM_SIZE_Z,
            sweep_threshold=SWEEP_THRES,
            score_column=SCORE_COLUMN,
        )
    )
    print(f"Finished processing tomogram: {i}")

df_predict = pd.concat(df_predicts, ignore_index=True)
df_predict.head()

Processing tomogram: test1
Finished processing tomogram: test1


,z,y,x,tomogram,label
0,29.0,364.876800,123.084708,test1,-1
1,34.0,446.906245,63.975754,test1,-1
2,34.0,425.605775,79.510368,test1,-1
3,37.0,712.295168,36.750520,test1,-1
4,38.0,548.490957,66.844897,test1,-1


## 4) Stage-2 3D CNN Classification and Center Refinement
Load the binary 3D CNN checkpoint and score each proposal crop.
The output `res` contains refined centers and classification confidence for downstream metrics/export.

In [ ]:
# Stage 2 classifier checkpoint (choose matching class + crop size).
# ribosome
CROP_SIZE = 65
ckpt_path = "<pathto>/release_model/ribosome/3DCNN.ckpt"
# hsp60
# CROP_SIZE = 41
# ckpt_path = "<pathto>/release_model/hsp60/3DCNN.ckpt"

cls_model = modules.ParticleID3DNet_Binary.load_from_checkpoint(ckpt_path)
if torch.cuda.is_available():
    device = torch.device("cuda:0")
else:
    device = torch.device("cpu")
cls_model = cls_model.to(device)

infer_ds = data.Particle3DDataset(
    df=df_predict,
    tomo_paths=tomo_paths,
    crop_size=CROP_SIZE,
    norm="hist",
    if_augmentation=False,
 )

res = utils.test_predict_df_with_revised_centers(
    df_predict=df_predict,
    dataset=infer_ds,
    model=cls_model,
    batch_size=32,
    center_method="weighted_average",
 )

res.head()

Preparing inference for 3097 candidates across 1 tomograms
Using batch_size=32, total_batches=97, center_method=weighted_average
Running model on device: cuda:0


Revising centers from predicted crop coordinates
Finished inference. Added prediction columns for 3097 rows.


,z,y,x,tomogram,label,prediction,prediction_score,revised_z,revised_y,revised_x
0,29.0,364.876800,123.084708,test1,-1,1,0.603129,35.651237,362.320452,122.684475
1,34.0,446.906245,63.975754,test1,-1,1,0.974559,42.148800,443.886097,66.138809
2,34.0,425.605775,79.510368,test1,-1,1,0.793203,38.987305,426.669151,78.081177
3,37.0,712.295168,36.750520,test1,-1,1,0.582068,35.464792,713.547874,33.806202
4,38.0,548.490957,66.844897,test1,-1,1,0.953628,45.440430,547.582964,68.929066


## 5) Evaluation (Adjusted AUPR)
Compute AUROC/AUPR-style metrics after optional sweep filtering and point matching to ground truth.
Use this section when labels are available for the same tomograms in `tomo_paths`.

In [63]:
# Evaluate `res` using adjusted AUPR (same style as train3DCNN copy.ipynb)
EVAL_SWEEP_THRES = SWEEP_THRES
EVAL_MATCH_THRES = 15

# Ground-truth centers ribosome
labels = np.loadtxt("<pathto>/release_model/data_example/ribosome_label.txt") 
labels = pd.DataFrame(labels, columns=["z", "y", "x"])  
labels["tomogram"] = "test1"

# Ground-truth centers hsp60
# labels = np.loadtxt("<pathto>/release_model/data_example/hsp60_label.txt") 
# labels = pd.DataFrame(labels, columns=["z", "y", "x"])  
# labels["tomogram"] = "test2"

pred_df = res.copy()
pred_df["tomogram"] = pred_df["tomogram"].astype(str)

per_tomo_rows = []
for tomo in tomo_paths:
    subdf = pred_df[pred_df["tomogram"] == tomo].copy()
    if len(subdf) == 0:
        continue

    score = subdf["prediction_score"].to_numpy(dtype=float)
    predict_center = subdf[["revised_z", "revised_y", "revised_x"]].to_numpy(dtype=float)

    # Keep the same evaluation process as train3DCNN copy: optional sweep before matching
    predict_center_f, score_f = utils.sweep_filter_points_by_distance(
        predict_center,
        score,
        threshold=EVAL_SWEEP_THRES,
    )
    
    sub_label = labels[labels["tomogram"] == tomo][["z", "y", "x"]].to_numpy()
    matches, match_dists, _ = postprocess.match_and_find_closest(predict_center_f, sub_label)
    t = postprocess.calculate_metrics(matches, match_dists, score_f, threshold=EVAL_MATCH_THRES)

    adjusted_aupr = np.nan
    if len(sub_label) > 0:
        adjusted_aupr = t["aupr2"] * t["cnts"] / len(sub_label)

    per_tomo_rows.append(
        {
            "tomogram": tomo,
            "n_labels": int(len(sub_label)),
            "n_candidates_before": int(len(score)),
            "n_candidates_after": int(len(score_f)),
            "auroc": float(t.get("auroc", np.nan)),
            "aupr": float(t.get("aupr", np.nan)),
            "cnts": float(t.get("cnts", np.nan)),
            "adjusted_aupr": float(adjusted_aupr),
        }
    )

    print(
        f"[EVAL] {tomo}: AUPR={t.get('aupr', np.nan):.6f}, "
        f"AUPR2={t.get('aupr2', np.nan):.6f}, Adjusted={adjusted_aupr:.6f}"
    )

aupr_table = pd.DataFrame(per_tomo_rows).sort_values("tomogram", ignore_index=True)
summary = pd.DataFrame(
    {
        "num_tomograms": [len(aupr_table)],
        "mean_aupr": [aupr_table["aupr"].mean() if len(aupr_table) else np.nan],
        "mean_adjusted_aupr": [aupr_table["adjusted_aupr"].mean() if len(aupr_table) else np.nan],
    }
)

print("\nAdjusted AUPR summary")
display(summary)
display(aupr_table)

[EVAL] test1: AUPR=0.778145, AUPR2=0.778722, Adjusted=0.749994

Adjusted AUPR summary


,num_tomograms,mean_aupr,mean_adjusted_aupr
0,1,0.778145,0.749994


,tomogram,n_labels,n_candidates_before,n_candidates_after,auroc,aupr,cnts,adjusted_aupr
0,test1,759,3097,2158,0.904854,0.778145,731.0,0.749994
